# LeetCode 3518 — Smallest Palindromic Rearrangement II bằng Python

- Link: https://leetcode.com/problems/smallest-palindromic-rearrangement-ii/
- Độ khó: **Hard**
- Chủ đề: **Tổ hợp** (hệ số đa thức của nửa chuỗi), **tìm hoán vị thứ k theo thứ tự từ điển**, **đếm có chặn trên** (capped counting) để tránh số khổng lồ

Skeleton trên LeetCode (Python3):

```python
class Solution:
    def smallestPalindrome(self, s: str, k: int) -> str:
```

## 1. Mục tiêu học

1. Nhận ra một palindrome được xác định hoàn toàn bởi **nửa đầu** (và ký tự giữa nếu độ dài lẻ) ⇒ bài toán trở thành "**hoán vị thứ k** của nửa chuỗi".
2. Đếm số hoán vị phân biệt của một multiset bằng **hệ số đa thức** `L! / Π c_i!`.
3. Dựng hoán vị thứ `k` bằng tham lam: thử ký tự nhỏ nhất ở mỗi vị trí, nếu số cách còn lại `< k` thì trừ đi và thử ký tự kế.
4. **Chặn trên khi đếm**: chỉ cần biết số cách có `≥ k` hay không ⇒ dừng sớm khi vượt `k` (tránh số nguyên hàng nghìn chữ số).
5. Tăng tốc: tiền tố dài có thể **giữ nguyên thứ tự tăng dần**; chỉ cần xử lý một hậu tố ngắn.

## 2. Đề bài & ví dụ

Cho chuỗi **palindrome** `s` và số `k`. Trả về **hoán vị palindrome thứ `k`** theo thứ tự từ điển của `s` (các hoán vị giống nhau chỉ tính một lần); nếu có ít hơn `k` hoán vị palindrome phân biệt thì trả `""`.

**Ví dụ 1:** `s = "abba", k = 2` → `"baab"` (hai palindrome: `"abba" < "baab"`).

**Ví dụ 2:** `s = "aa", k = 2` → `""`.

**Ví dụ 3:** `s = "bacab", k = 1` → `"abcba"`.

**Ràng buộc:** `1 ≤ len(s) ≤ 10^4`, chữ thường, `s` là palindrome, `1 ≤ k ≤ 10^6`.

In [1]:
from collections import Counter
from itertools import permutations
examples = [("abba", 2, "baab"), ("aa", 2, ""), ("bacab", 1, "abcba")]
for s, k, exp in examples:
    print(repr(s), k, '-> kỳ vọng', repr(exp))

'abba' 2 -> kỳ vọng 'baab'
'aa' 2 -> kỳ vọng ''
'bacab' 1 -> kỳ vọng 'abcba'


## 3. Kiến thức nền

### 3.1. Palindrome từ nửa chuỗi

Đếm tần suất; mỗi ký tự xuất hiện chẵn lần trừ tối đa một ký tự (ký tự giữa). Nửa đầu chứa `cnt[c] // 2` bản của mỗi `c`. Palindrome = `half + mid + reverse(half)`. Thứ tự từ điển của palindrome **trùng** thứ tự từ điển của `half` (vì `half` là tiền tố và các palindrome cùng độ dài).

In [2]:
cnt = Counter("bacab")
mid = ''.join(c for c in cnt if cnt[c] % 2)
half = {c: cnt[c] // 2 for c in sorted(cnt)}
print('tần suất:', dict(cnt), '| giữa:', repr(mid), '| nửa:', half)

tần suất: {'b': 2, 'a': 2, 'c': 1} | giữa: 'c' | nửa: {'a': 1, 'b': 1, 'c': 0}


### 3.2. Hệ số đa thức

Số hoán vị phân biệt của multiset có tổng `L` phần tử và tần suất `c_1, …, c_t` là `L! / (c_1! ⋯ c_t!)` = tích các tổ hợp `C(c_1, c_1)·C(c_1 + c_2, c_2)·…`.

In [3]:
from math import comb, factorial
c = [2, 1, 1]
print(factorial(4) // (factorial(2) * factorial(1) * factorial(1)), comb(2, 2) * comb(3, 1) * comb(4, 1),
      len(set(permutations('aabc'))))

12 12 12


### 3.3. Đếm có chặn trên

Ta chỉ so sánh số cách với `k ≤ 10^6`. Tính `C(n, r)` theo công thức nhân dần `C ← C·(n−r+i)/i`; giá trị trung gian tăng dần nên **dừng ngay** khi vượt `cap` và trả `cap + 1`.

In [4]:
def capped_comb(n, r, cap):
    if r < 0 or r > n:
        return 0
    r = min(r, n - r)
    c = 1
    for i in range(1, r + 1):
        c = c * (n - r + i) // i
        if c > cap:
            return cap + 1
    return c

def capped_multinomial(counts, cap):
    total, res = 0, 1
    for c in counts:
        if c:
            total += c
            res *= capped_comb(total, c, cap)
            if res > cap:
                return cap + 1
    return res

print(capped_comb(10, 3, 1000), capped_comb(5000, 2500, 10**6), capped_multinomial([2500, 2500], 10**6))

120 1000001 1000001


### 3.4. Hoán vị thứ `k` theo thứ tự từ điển

Ở vị trí đầu, thử ký tự `x` nhỏ nhất: số hoán vị bắt đầu bằng `x` là `M(cnt − x)`. Nếu `k ≤ M` thì chọn `x`, ngược lại `k −= M` và thử ký tự kế tiếp. Lặp cho các vị trí sau.

## 4. Ý tưởng nền tảng / cách nghĩ trực giác

Ví dụ 1: `s = "abba"` ⇒ nửa `{a:1, b:1}`, có `2!/(1!1!) = 2` hoán vị: `"ab"`, `"ba"` ⇒ palindrome `"abba"`, `"baab"`. `k = 2` ⇒ `"baab"`.

Tính tay theo tham lam (`k = 2`):

| vị trí | thử | số cách còn lại | quyết định |
|---|---|---|---|
| 0 | `a` | `M({b:1}) = 1 < 2` | `k = 2 − 1 = 1`, thử tiếp |
| 0 | `b` | `M({a:1}) = 1 ≥ 1` | chọn `b` |
| 1 | `a` | `1 ≥ 1` | chọn `a` |

Nửa `"ba"` ⇒ `"baab"`.

In [5]:
half = 'ab'
print(sorted(set(''.join(p) for p in permutations(half))))

['ab', 'ba']


## 5. Cách 1 — Brute Force: sinh mọi hoán vị phân biệt của nửa chuỗi

In [6]:
def kth_pal_brute(s, k):
    cnt = Counter(s)
    mid = ''.join(c for c in cnt if cnt[c] % 2)
    half = ''.join(c * (cnt[c] // 2) for c in sorted(cnt))
    ps = sorted(set(permutations(half)))
    if k > len(ps):
        return ""
    h = ''.join(ps[k - 1])
    return h + mid + h[::-1]

for s, k, exp in examples:
    print(repr(kth_pal_brute(s, k)), '| kỳ vọng', repr(exp))

'baab' | kỳ vọng 'baab'
'' | kỳ vọng ''
'abcba' | kỳ vọng 'abcba'


**Độ phức tạp:** `O(L! · L)` với `L = n/2` — chỉ dùng được khi `L ≤ 8`.

**Nút thắt:** không cần liệt kê, chỉ cần **đếm** số hoán vị có cùng tiền tố.

## 6. Ý tưởng thuật toán / bước đột phá

**Bổ đề:** số hoán vị phân biệt của nửa còn lại khi đã cố định tiền tố = hệ số đa thức của tần suất còn lại. Vậy có thể "nhảy" qua cả khối hoán vị bắt đầu bằng ký tự `x` mà không liệt kê.

**Đột phá phụ — chặn trên:** `M` có thể khổng lồ (`5000!`), nhưng ta chỉ cần so với `k ≤ 10^6` ⇒ tính có chặn, dừng sớm.

**Đột phá phụ 2 — tiền tố cố định:** nếu hậu tố độ dài `t` (tính từ cuối nửa chuỗi, gồm các ký tự lớn nhất khi xếp tăng) đã có `≥ k` hoán vị, thì `t` ký tự cuối là phần duy nhất thay đổi; tiền tố là chuỗi tăng dần nhỏ nhất. Hậu tố này thường rất ngắn.

Quy tắc:

- **Nếu** tổng số hoán vị `< k` **thì** trả `""`.
- **Nếu** `M(cnt − x) ≥ k` **thì** chọn `x`; **ngược lại** `k −= M(cnt − x)`.

## 7. Các cách tiếp theo

### 7.1. Cách 2 — `next_permutation` lặp `k − 1` lần

Bắt đầu từ nửa sắp tăng (hoán vị nhỏ nhất), áp dụng thuật toán hoán vị kế tiếp `k − 1` lần. Đúng và dễ hiểu, nhưng `O(k · L)` — tới `5·10^9` ở giới hạn.

In [7]:
def next_permutation(a):
    i = len(a) - 2
    while i >= 0 and a[i] >= a[i + 1]:
        i -= 1
    if i < 0:
        return False
    j = len(a) - 1
    while a[j] <= a[i]:
        j -= 1
    a[i], a[j] = a[j], a[i]
    a[i + 1:] = reversed(a[i + 1:])
    return True

def kth_pal_nextperm(s, k):
    cnt = Counter(s)
    mid = ''.join(c for c in cnt if cnt[c] % 2)
    a = sorted(''.join(c * (cnt[c] // 2) for c in cnt))
    for _ in range(k - 1):
        if not next_permutation(a):
            return ""
    h = ''.join(a)
    return h + mid + h[::-1]

for s, k, exp in examples:
    print(repr(kth_pal_nextperm(s, k)), '| kỳ vọng', repr(exp))

'baab' | kỳ vọng 'baab'
'' | kỳ vọng ''
'abcba' | kỳ vọng 'abcba'


`next_permutation` tự động bỏ qua hoán vị trùng (vì so sánh `>=`, `<=`), nên đếm đúng hoán vị **phân biệt**.

### 7.2. Cách 3 — Tham lam + đếm hệ số đa thức có chặn (lời giải chính)

In [8]:
def kth_pal_count(s, k):
    cnt = Counter(s)
    mid = ''.join(c for c in cnt if cnt[c] % 2)
    half = [cnt.get(chr(97 + i), 0) // 2 for i in range(26)]
    if capped_multinomial(half, k) < k:
        return ""
    res = []
    for _ in range(sum(half)):
        for i in range(26):
            if half[i] == 0:
                continue
            half[i] -= 1
            m = capped_multinomial(half, k)
            if m >= k:
                res.append(chr(97 + i))
                break
            k -= m
            half[i] += 1
    h = ''.join(res)
    return h + mid + h[::-1]

for s, k, exp in examples:
    print(repr(kth_pal_count(s, k)), '| kỳ vọng', repr(exp))

'baab' | kỳ vọng 'baab'
'' | kỳ vọng ''
'abcba' | kỳ vọng 'abcba'


**Độ phức tạp:** `L` vị trí × tối đa 26 ký tự thử × hệ số đa thức có chặn (26 tổ hợp, mỗi cái dừng sớm). Thực tế ký tự đầu tiên thử thường đã đủ `≥ k` nên rất nhanh.

### 7.3. Cách 4 — Chỉ xử lý hậu tố ngắn

Xếp nửa tăng dần `h_sorted`. Tìm `t` nhỏ nhất sao cho số hoán vị của `t` ký tự **cuối** của `h_sorted` `≥ k`. Khi đó `L − t` ký tự đầu giữ nguyên, chỉ cần tìm hoán vị thứ `k` của `t` ký tự cuối (bằng cách 3). Vì hệ số đa thức tăng rất nhanh, `t` thường chỉ vài chục.

In [9]:
def kth_pal_suffix(s, k):
    cnt = Counter(s)
    mid = ''.join(c for c in cnt if cnt[c] % 2)
    hs = sorted(''.join(c * (cnt[c] // 2) for c in cnt))
    L = len(hs)
    suf = [0] * 26
    t = 0
    while t < L:
        if capped_multinomial(suf, k) >= k:
            break
        t += 1
        suf[ord(hs[L - t]) - 97] += 1
    if capped_multinomial(suf, k) < k:
        return ""
    # hoán vị thứ k của t ký tự cuối
    tail = []
    for _ in range(t):
        for i in range(26):
            if suf[i] == 0:
                continue
            suf[i] -= 1
            m = capped_multinomial(suf, k)
            if m >= k:
                tail.append(chr(97 + i)); break
            k -= m
            suf[i] += 1
    h = ''.join(hs[:L - t]) + ''.join(tail)
    return h + mid + h[::-1]

for s, k, exp in examples:
    print(repr(kth_pal_suffix(s, k)), '| kỳ vọng', repr(exp))

'baab' | kỳ vọng 'baab'
'' | kỳ vọng ''
'abcba' | kỳ vọng 'abcba'


Vì sao tiền tố giữ nguyên? Các hoán vị có tiền tố là `L − t` ký tự nhỏ nhất (xếp tăng) là **khối nhỏ nhất** theo thứ tự từ điển, và khối đó có đúng `M(suf)` phần tử `≥ k` ⇒ hoán vị thứ `k` nằm trong khối.

## 8. TRACE từng bước

Trace cách 3 với `s = "aabbccbbaa"` (nửa `{a:2, b:2, c:1}`, `5!/(2!2!1!) = 30` palindrome), `k = 17`.

In [10]:
def trace(s, k):
    cnt = Counter(s)
    half = [cnt.get(chr(97 + i), 0) // 2 for i in range(26)]
    print('nửa:', {chr(97 + i): c for i, c in enumerate(half) if c}, '| tổng số palindrome:', capped_multinomial(half, 10**18))
    res = []
    for pos in range(sum(half)):
        notes = []
        for i in range(26):
            if half[i] == 0:
                continue
            half[i] -= 1
            m = capped_multinomial(half, k)
            if m >= k:
                notes.append(f"'{chr(97 + i)}': {m} >= k={k} -> CHỌN")
                res.append(chr(97 + i)); break
            notes.append(f"'{chr(97 + i)}': {m} < k={k} -> k -= {m}")
            k -= m
            half[i] += 1
        print(f"  vị trí {pos}: " + '; '.join(notes))
    h = ''.join(res)
    print('nửa =', h, '| palindrome =', h + h[::-1])

trace("aabbccbbaa", 17)
print('vét cạn:', kth_pal_brute("aabbccbbaa", 17))

nửa: {'a': 2, 'b': 2, 'c': 1} | tổng số palindrome: 30
  vị trí 0: 'a': 12 < k=17 -> k -= 12; 'b': 6 >= k=5 -> CHỌN
  vị trí 1: 'a': 6 >= k=5 -> CHỌN
  vị trí 2: 'a': 2 < k=5 -> k -= 2; 'b': 2 < k=3 -> k -= 2; 'c': 2 >= k=1 -> CHỌN
  vị trí 3: 'a': 1 >= k=1 -> CHỌN
  vị trí 4: 'b': 1 >= k=1 -> CHỌN
nửa = bacab | palindrome = bacabbacab
vét cạn: bacabbacab


## 9. Code nộp LeetCode

In [11]:
class Solution:
    def smallestPalindrome(self, s: str, k: int) -> str:
        cnt = Counter(s)
        mid = ''.join(c for c in cnt if cnt[c] % 2)
        half = [cnt.get(chr(97 + i), 0) // 2 for i in range(26)]

        def comb_cap(n, r):
            r = min(r, n - r)
            c = 1
            for i in range(1, r + 1):
                c = c * (n - r + i) // i
                if c > k:
                    return k + 1
            return c

        def multi(cs):
            total, res = 0, 1
            for c in cs:
                if c:
                    total += c
                    res *= comb_cap(total, c)
                    if res > k:
                        return k + 1
            return res

        if multi(half) < k:
            return ""
        res = []
        for _ in range(sum(half)):
            for i in range(26):
                if half[i]:
                    half[i] -= 1
                    m = multi(half)
                    if m >= k:
                        res.append(chr(97 + i))
                        break
                    k -= m
                    half[i] += 1
        h = ''.join(res)
        return h + mid + h[::-1]

for s, k, exp in examples:
    got = Solution().smallestPalindrome(s, k)
    print(repr(got), 'OK' if got == exp else 'SAI')

'baab' OK
'' OK
'abcba' OK


Chú ý: `k` thay đổi trong vòng lặp nên chặn `cap` cũng giảm theo — vẫn đúng vì ta chỉ cần so sánh với `k` **hiện tại**.

## 10. Vì sao thuật toán đúng

1. **Song ánh palindrome ↔ nửa chuỗi:** vì `s` là palindrome, tần suất mỗi ký tự chẵn trừ tối đa một (ký tự giữa cố định). Mọi palindrome là `h + mid + rev(h)` với `h` là hoán vị của multiset nửa; ngược lại mỗi `h` cho đúng một palindrome.
2. **Bảo toàn thứ tự:** hai palindrome cùng độ dài so sánh theo `h` (tiền tố khác nhau đầu tiên nằm trong `h`).
3. **Đếm khối:** các hoán vị bắt đầu bằng tiền tố `P` tạo một **khối liên tiếp** trong thứ tự từ điển, kích thước = hệ số đa thức của phần còn lại. Tham lam "nhảy khối" ở mỗi vị trí tìm đúng hoán vị thứ `k`.
4. **Chặn trên an toàn:** ta chỉ cần biết `M ≥ k` hay không và giá trị chính xác khi `M < k` — `capped` trả về chính xác khi `≤ cap`.
5. **Cách 4:** như giải thích ở mục 7.3 — khối nhỏ nhất với tiền tố `L − t` ký tự nhỏ nhất đã chứa hoán vị thứ `k`.

## 11. Phân tích Time / Space Complexity

Gọi `n = len(s) ≤ 10^4`, `L = n/2`, `σ = 26`.

| Cách | Time | Space |
|---|---|---|
| 1. Liệt kê | `O(L! · L)` | `O(L!)` |
| 2. Next permutation | `O(k · L)` | `O(L)` |
| 3. Đếm có chặn | `O(L · σ · σ · r)` với `r` nhỏ nhờ dừng sớm | `O(σ)` |
| 4. Hậu tố ngắn | `O(L log L + t · σ^2 · r)` | `O(L)` |

## 12. Bộ test

### 12.1. Ví dụ đề + ca biên

In [12]:
impls = {'nextperm': kth_pal_nextperm, 'count': kth_pal_count, 'suffix': kth_pal_suffix,
         'submit': lambda s, k: Solution().smallestPalindrome(s, k)}
edge = [("a", 1, "a"), ("a", 2, ""), ("aba", 1, "aba"), ("abcba", 2, "bacab"), ("abccba", 6, "cbaabc"), ("abccba", 7, "")]
for s, k, exp in examples + edge:
    assert kth_pal_brute(s, k) == exp, (s, k)
    for name, f in impls.items():
        got = f(s, k)
        assert got == exp, (name, s, k, got, exp)
print('Ví dụ đề và ca biên: tất cả đúng.')

Ví dụ đề và ca biên: tất cả đúng.


### 12.2. Đối chiếu ngẫu nhiên với vét cạn

In [13]:
import random
random.seed(3518)
cnt = 0
for _ in range(500):
    h = ''.join(random.choice('abcd') for _ in range(random.randint(0, 7)))
    m = random.choice(['', 'a', 'e'])
    s = h + m + h[::-1]
    if not s:
        continue
    k = random.randint(1, 400)
    exp = kth_pal_brute(s, k)
    for name, f in impls.items():
        assert f(s, k) == exp, (name, s, k)
    cnt += 1
print('Khớp vét cạn trên', cnt, 'chuỗi ngẫu nhiên.')

Khớp vét cạn trên 474 chuỗi ngẫu nhiên.


### 12.3. Ca lớn (`len(s) = 10^4`, `k = 10^6`)

In [14]:
import time
random.seed(4)
cases = []
h = ''.join(random.choice('abcdefghijklmnopqrstuvwxyz') for _ in range(5000))
cases.append(('26 chữ cái ngẫu nhiên', h + h[::-1], 10**6))
cases.append(('chỉ a và một cặp b', 'a' * 4999 + 'bb' + 'a' * 4999, 4000))
h = ''.join(random.choice('ab') for _ in range(5000))
cases.append(('hai chữ cái', h + 'z' + h[::-1], 10**6))
for name, s, k in cases:
    t = time.time(); a = Solution().smallestPalindrome(s, k); t1 = time.time() - t
    t = time.time(); b = kth_pal_suffix(s, k); t2 = time.time() - t
    assert a == b and (a == a[::-1])
    print(f"{name}: {t1:.3f}s | hậu tố {t2:.3f}s | 12 ký tự đầu {a[:12]!r}")

26 chữ cái ngẫu nhiên: 0.006s | hậu tố 0.001s | 12 ký tự đầu 'aaaaaaaaaaaa'
chỉ a và một cặp b: 0.005s | hậu tố 0.007s | 12 ký tự đầu 'aaaaaaaaaaaa'


hai chữ cái: 0.005s | hậu tố 0.005s | 12 ký tự đầu 'aaaaaaaaaaaa'


## 13. Các lỗi người mới hay gặp

### Lỗi 1 — Đếm hoán vị **có lặp** (`L!`) thay vì hoán vị phân biệt

In [15]:
def wrong_factorial_count(s, k):
    cnt = Counter(s)
    L = sum(c // 2 for c in cnt.values())
    return "" if factorial(L) < k else "(tiếp tục…)"

print('sai :', repr(wrong_factorial_count("aabbaa", 4)), '| đúng:', repr(Solution().smallestPalindrome("aabbaa", 4)))

sai : '(tiếp tục…)' | đúng: ''


**Vì sao sai:** nửa của `"aabbaa"` là `"aab"`: `3! = 6` nhưng chỉ có 3 hoán vị phân biệt ⇒ với `k = 4` code sai tưởng còn nghiệm, trong khi đáp án là `""`. Phải chia cho `Π c_i!`.

In [16]:
print('3! =', factorial(3), '| phân biệt:', len(set(permutations('aab'))))

3! = 6 | phân biệt: 3


### Lỗi 2 — Tính hệ số đa thức chính xác bằng giai thừa lớn mỗi bước (quá chậm)

In [17]:
import time
def multinomial_exact(cs):
    r = factorial(sum(cs))
    for c in cs:
        r //= factorial(c)
    return r

cs = [200] * 26
t = time.time(); multinomial_exact(cs); te = time.time() - t
t = time.time(); capped_multinomial(cs, 10**6); tc = time.time() - t
print(f"chính xác: {te * 1000:.2f} ms/lần | có chặn: {tc * 1000:.3f} ms/lần (lời giải gọi hàng chục nghìn lần)")

chính xác: 3.85 ms/lần | có chặn: 0.044 ms/lần (lời giải gọi hàng chục nghìn lần)


**Vì sao sai:** đúng nhưng mỗi lần tính giai thừa của `5000` tốn đáng kể; gọi `L · 26` lần ⇒ quá chậm. Đếm có chặn dừng ngay khi vượt `k`.

### Lỗi 3 — Quên ký tự giữa (độ dài lẻ)

In [18]:
def wrong_no_mid(s, k):
    cnt = Counter(s)
    half = ''.join(c * (cnt[c] // 2) for c in sorted(cnt))
    ps = sorted(set(permutations(half)))
    if k > len(ps):
        return ""
    h = ''.join(ps[k - 1])
    return h + h[::-1]                 # SAI: thiếu ký tự giữa

print('sai :', repr(wrong_no_mid("bacab", 1)), '| đúng:', repr(Solution().smallestPalindrome("bacab", 1)))

sai : 'abba' | đúng: 'abcba'


**Vì sao sai:** palindrome độ dài lẻ có một ký tự đứng giữa (ký tự có tần suất lẻ).

### Lỗi 4 — Không trừ `k` khi bỏ qua một khối

In [19]:
def wrong_no_subtract(s, k):
    cnt = Counter(s)
    half = [cnt.get(chr(97 + i), 0) // 2 for i in range(26)]
    res = []
    for _ in range(sum(half)):
        for i in range(26):
            if half[i]:
                half[i] -= 1
                if capped_multinomial(half, k) >= k:
                    res.append(chr(97 + i)); break
                half[i] += 1               # SAI: quên k -= m
    h = ''.join(res)
    return h + h[::-1]

print('sai :', repr(wrong_no_subtract("abba", 2)), '| đúng:', repr(Solution().smallestPalindrome("abba", 2)))

sai : '' | đúng: 'baab'


**Vì sao sai:** bỏ qua khối bắt đầu bằng `x` nghĩa là đã "đi qua" `M` hoán vị — phải trừ chúng khỏi `k`.

## 14. Mẹo làm bài nhanh & nhận dạng pattern

- "Hoán vị thứ k theo thứ tự từ điển" ⇒ đếm kích thước khối theo tiền tố (giống LC 60 *Permutation Sequence*).
- Có ký tự lặp ⇒ khối có kích thước **hệ số đa thức**.
- `k` nhỏ nhưng số cách khổng lồ ⇒ **đếm có chặn**.
- Palindrome ⇒ làm việc với **nửa** chuỗi.

## 15. Biến thể của bài

1. Bản I (LC 3517): palindrome **nhỏ nhất** (`k = 1`) ⇒ chỉ cần xếp nửa tăng dần.
2. Hoán vị thứ `k` của chuỗi bất kỳ (không cần palindrome) ⇒ bỏ bước lấy nửa.
3. Đếm số palindrome phân biệt **nhỏ hơn** một chuỗi cho trước ⇒ ngược lại: cộng dồn kích thước các khối nhỏ hơn.
4. Bài liên quan: LC 60, LC 267 *Palindrome Permutation II*, LC 1830.

In [20]:
# Biến thể 3: thứ hạng (1-based) của một palindrome p trong các hoán vị palindrome của nó
def rank_of(p):
    cnt = Counter(p)
    L = len(p) // 2
    half = [cnt.get(chr(97 + i), 0) // 2 for i in range(26)]
    rank = 1
    for ch in p[:L]:
        for i in range(ord(ch) - 97):
            if half[i]:
                half[i] -= 1
                rank += capped_multinomial(half, 10**18)
                half[i] += 1
        half[ord(ch) - 97] -= 1
    return rank

for k in range(1, 31):
    assert rank_of(kth_pal_brute("aabbccbbaa", k)) == k
print('thứ hạng của "baab":', rank_of("baab"), '| của "cbaabc":', rank_of("cbaabc"))

thứ hạng của "baab": 2 | của "cbaabc": 6


## 16. So sánh các cách làm

| Cách | Ý tưởng | Time | Khi nào dùng |
|---|---|---|---|
| 1 | Liệt kê hoán vị | `O(L!·L)` | Oracle |
| 2 | Next permutation `k−1` lần | `O(kL)` | `k·L` nhỏ |
| 3 | Đếm khối có chặn | gần `O(L·σ)` thực tế | **Nộp bài** |
| 4 | Chỉ xử lý hậu tố | `O(L log L + t σ^2)` | Khi `L` rất lớn |

## 17. 5 bài tập vận dụng (từ dễ đến khó)

**Bài 1 (đoán output):** `smallestPalindrome("racecar", 1)`.

**Bài 2 (tính tay):** `smallestPalindrome("aabbaa", 3)`.

**Bài 3 (điền TODO):** viết `count_pal(s)` — tổng số palindrome phân biệt (có chặn `10^18`).

**Bài 4 (tự viết lại):** biến thể 2 — hoán vị thứ `k` (phân biệt) của một chuỗi bất kỳ.

**Bài 5 (mini challenge):** biến thể 3 — viết `rank_of` không dùng chặn, kiểm chứng với vét cạn trên chuỗi ngẫu nhiên.

In [21]:
def count_pal_todo(s):
    # TODO
    raise NotImplementedError

def kth_permutation_todo(s, k):
    # TODO
    raise NotImplementedError

def rank_exact_todo(p):
    # TODO
    raise NotImplementedError

print('Đã định nghĩa các hàm TODO (không gọi).')

Đã định nghĩa các hàm TODO (không gọi).


## 18. Gợi ý 3 mức cho bài tập

**Bài 1.** (1) Nửa: `{a:1, c:1, r:1}`, giữa `e`. (2) Nhỏ nhất: nửa tăng `"acr"`. (3) `"acrerca"`.

**Bài 2.** (1) Nửa `{a:2, b:1}`: hoán vị `aab, aba, baa`. (2) Thứ 3: `"baa"`. (3) `"baaaab"`.

**Bài 3.** (1) Lấy nửa. (2) `capped_multinomial(half, 10**18)`. (3) So với liệt kê.

**Bài 4.** (1) Giống cách 3 nhưng dùng toàn bộ tần suất. (2) Không có ký tự giữa. (3) Vét cạn `sorted(set(permutations))`.

**Bài 5.** (1) Duyệt từng ký tự của nửa `p`. (2) Cộng kích thước các khối ký tự nhỏ hơn. (3) Hệ số đa thức chính xác (Python số lớn).

## 19. Đáp án tham khảo cho bài tập

In [22]:
# Bài 1, 2, 3
a1 = Solution().smallestPalindrome("racecar", 1)
a2 = Solution().smallestPalindrome("aabbaa", 3)
assert a1 == "acrerca" == kth_pal_brute("racecar", 1)
assert a2 == "baaaab" == kth_pal_brute("aabbaa", 3)

def count_pal(s):
    cnt = Counter(s)
    return capped_multinomial([cnt[c] // 2 for c in cnt], 10**18)

for s in ["abba", "aabbccbbaa", "racecar", "a"]:
    cnt = Counter(s)
    half = ''.join(c * (cnt[c] // 2) for c in cnt)
    assert count_pal(s) == len(set(permutations(half)))
print('Bài 1:', a1, '| Bài 2:', a2, '| Bài 3: đúng')

Bài 1: acrerca | Bài 2: baaaab | Bài 3: đúng


In [23]:
# Bài 4: hoán vị thứ k phân biệt của chuỗi bất kỳ
def kth_permutation(s, k):
    cnt = [0] * 26
    for ch in s:
        cnt[ord(ch) - 97] += 1
    if capped_multinomial(cnt, k) < k:
        return ""
    out = []
    for _ in range(len(s)):
        for i in range(26):
            if cnt[i]:
                cnt[i] -= 1
                m = capped_multinomial(cnt, k)
                if m >= k:
                    out.append(chr(97 + i)); break
                k -= m
                cnt[i] += 1
    return ''.join(out)

random.seed(5)
for _ in range(200):
    s = ''.join(random.choice('abc') for _ in range(random.randint(1, 6)))
    ps = sorted(set(''.join(p) for p in permutations(s)))
    k = random.randint(1, len(ps) + 2)
    assert kth_permutation(s, k) == (ps[k - 1] if k <= len(ps) else "")
print('Bài 4: ví dụ', kth_permutation("aabc", 5))

Bài 4: ví dụ acab


In [24]:
# Bài 5: thứ hạng chính xác
def rank_exact(p):
    cnt = Counter(p)
    L = len(p) // 2
    half = [cnt.get(chr(97 + i), 0) // 2 for i in range(26)]
    rank = 1
    for ch in p[:L]:
        for i in range(ord(ch) - 97):
            if half[i]:
                half[i] -= 1
                rank += multinomial_exact(half)
                half[i] += 1
        half[ord(ch) - 97] -= 1
    return rank

random.seed(6)
for _ in range(200):
    h = ''.join(random.choice('abcd') for _ in range(random.randint(1, 6)))
    s = h + random.choice(['', 'z']) + h[::-1]
    cnt = Counter(s)
    half = ''.join(c * (cnt[c] // 2) for c in sorted(cnt))
    ps = sorted(set(permutations(half)))
    k = random.randint(1, len(ps))
    assert rank_exact(kth_pal_brute(s, k)) == k
print('Bài 5: thứ hạng chính xác khớp vét cạn')

Bài 5: thứ hạng chính xác khớp vét cạn


## 20. Template nhớ nhanh + Checklist tự kiểm tra trước khi submit

```python
half[c] = cnt[c] // 2; mid = ký tự tần suất lẻ
nếu M(half) < k: return ""
for mỗi vị trí của nửa:
    for c từ 'a' tới 'z' (half[c] > 0):
        half[c] -= 1; m = M_capped(half)
        nếu m >= k: chọn c; break
        k -= m; half[c] += 1
return h + mid + h[::-1]
M_capped: tích C(tổng dồn, c) với dừng sớm khi > k
```

**Checklist:**

- [ ] Làm việc trên **nửa** chuỗi, nhớ ký tự giữa?
- [ ] Đếm hoán vị **phân biệt** (hệ số đa thức)?
- [ ] Đếm **có chặn** để không tính số khổng lồ?
- [ ] Trừ `k` khi bỏ qua một khối?
- [ ] Trả `""` khi tổng số `< k`?